In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.8 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
DATA_DIR = "./icenet"
YOLO_DATA_DIR = "./yolo_icenet"

CLASS_LIST = ["ice"]

In [ ]:
def list_directory_tree(start_path):
    print(f"\n--- Directory Tree from {start_path} ---")
    if not os.path.exists(start_path):
        print(f"Error: Path not found - {start_path}")
        return

    for root, dirs, files in os.walk(start_path):
        level = root.replace(start_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')
    print("---------------------------------------")


In [ ]:
import os
import glob

def get_paths(data_dir, split):
    """Function to get image and mask paths for a given split."""
    split_dir = os.path.join(data_dir, split)

    image_paths = sorted([
        os.path.join(split_dir, f)
        for f in os.listdir(split_dir)
        if f.endswith(('.jpg', '.png')) and '_mask' not in f
    ])

    mask_paths = sorted([
        os.path.join(split_dir, f)
        for f in os.listdir(split_dir)
        if f.endswith('_mask.png')
    ])

    if len(image_paths) != len(mask_paths):
        print(f"Warning: Number of images and masks do not match for {split} split!")
        print(f"  Images found: {len(image_paths)}")
        print(f"  Masks found: {len(mask_paths)}")

    return image_paths, mask_paths

train_images, train_masks = get_paths(DATA_DIR, 'train')
val_images, val_masks = get_paths(DATA_DIR, 'valid')
test_images, test_masks = get_paths(DATA_DIR, 'test')

print(f"Training set: {len(train_images)} images")
print(f"Validation set: {len(val_images)} images")
print(f"Test set: {len(test_images)} images")

Training set: 478 images
Validation set: 154 images
Test set: 153 images


In [ ]:
# Part 2: Data Conversion from PNG Masks to YOLO Format

def convert_mask_to_yolo_segmentation(mask_path, save_path, class_id):
    """Converts a single PNG mask to YOLO segmentation format."""
    try:
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if mask is None or np.sum(mask) == 0:
            print(f"Warning: Skipping {os.path.basename(mask_path)} as it's an empty or invalid mask.")
            return

        h, w = mask.shape
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        yolo_lines = []
        for contour in contours:
            normalized_points = []
            for point in contour.squeeze():
                x, y = point
                normalized_points.append(x / w)
                normalized_points.append(y / h)

            flattened_points = normalized_points

            yolo_line = f"{class_id} " + " ".join(map(str, flattened_points))
            yolo_lines.append(yolo_line)

        if yolo_lines:
            with open(save_path, 'w') as f:
                f.write("\n".join(yolo_lines))
        else:
            print(f"Warning: No contours found in mask {os.path.basename(mask_path)}. Not creating a label file.")

    except Exception as e:
        print(f"Error processing {os.path.basename(mask_path)}: {e}")

list_directory_tree(DATA_DIR)

print("\nStarting data conversion...")

train_images, train_masks = get_paths(DATA_DIR, 'train')
val_images, val_masks = get_paths(DATA_DIR, 'valid')
test_images, test_masks = get_paths(DATA_DIR, 'test')

paths = {
    'train': {'images': train_images, 'masks': train_masks},
    'valid': {'images': val_images, 'masks': val_masks},
    'test': {'images': test_images, 'masks': test_masks}
}

if os.path.exists(YOLO_DATA_DIR):
    shutil.rmtree(YOLO_DATA_DIR)

for split, split_paths in paths.items():
    img_paths = split_paths['images']
    mask_paths = split_paths['masks']

    yolo_img_dir = os.path.join(YOLO_DATA_DIR, 'images', split)
    yolo_ann_dir = os.path.join(YOLO_DATA_DIR, 'labels', split)
    os.makedirs(yolo_img_dir, exist_ok=True)
    os.makedirs(yolo_ann_dir, exist_ok=True)

    print(f"Converting {len(img_paths)} images for {split} split...")
    for img_path in tqdm(img_paths, desc=f"Converting {split} set"):
        img_name = os.path.basename(img_path)
        base_name = os.path.splitext(img_name)[0]

        mask_name = base_name + '_mask.png'
        mask_path = os.path.join(DATA_DIR, split, mask_name)

        if not os.path.exists(mask_path):
            print(f"Warning: No mask found for {img_name}. Skipping.")
            continue

        shutil.copyfile(img_path, os.path.join(yolo_img_dir, img_name))

        save_path = os.path.join(yolo_ann_dir, base_name + '.txt')
        convert_mask_to_yolo_segmentation(mask_path, save_path, CLASS_LIST.index("ice"))

print("Data conversion complete!")


--- Directory Tree from ./icenet ---
icenet/
    README.dataset.txt
    README.roboflow.txt
    test/
        0622_1050_jpg.rf.866519451400deaaf48d165254404a9e.jpg
        0622_1050_jpg.rf.866519451400deaaf48d165254404a9e_mask.png
        0622_1063_jpg.rf.ca23de7eb7b4c04bf11fe1f6c10e7112.jpg
        0622_1063_jpg.rf.ca23de7eb7b4c04bf11fe1f6c10e7112_mask.png
        0622_1218_jpg.rf.97f9bbe4335cfd5871c058b6f1c7d820.jpg
        0622_1218_jpg.rf.97f9bbe4335cfd5871c058b6f1c7d820_mask.png
        0622_1307_jpg.rf.85d20ae7afc9dc875069b7df1e01df78.jpg
        0622_1307_jpg.rf.85d20ae7afc9dc875069b7df1e01df78_mask.png
        0622_1633_jpg.rf.5812c1052f60b82f1fb2aa1f1c8195a0.jpg
        0622_1633_jpg.rf.5812c1052f60b82f1fb2aa1f1c8195a0_mask.png
        0622_16387_jpg.rf.204730bb4a05c34e83c410b1a676f56b.jpg
        0622_16387_jpg.rf.204730bb4a05c34e83c410b1a676f56b_mask.png
        0622_16431_jpg.rf.e31ec93cb2c12c592950d8ef145e2da5.jpg
        0622_16431_jpg.rf.e31ec93cb2c12c592950d8ef145e2da5

Converting train set:  31%|███       | 148/478 [00:00<00:00, 357.86it/s]

Error processing 542512_png_jpg.rf.0eb5e4e2b89c11c161c3cd49391d4ab1_mask.png: cannot unpack non-iterable numpy.int32 object


Converting train set:  46%|████▌     | 221/478 [00:00<00:00, 350.55it/s]

Converting train set:  70%|██████▉   | 333/478 [00:00<00:00, 340.86it/s]

Converting train set:  91%|█████████ | 436/478 [00:01<00:00, 334.04it/s]

... (4 outputs left out)

Converting 153 images for test split...


Converting test set:  28%|██▊       | 43/153 [00:00<00:00, 427.97it/s]

Converting test set: 100%|██████████| 153/153 [00:00<00:00, 393.06it/s]

Data conversion complete!


In [ ]:
# Part 3: YOLO Configuration File (data.yaml)
data_yaml_content = f"""
train: {YOLO_DATA_DIR}/images/train/
val: {YOLO_DATA_DIR}/images/valid/
test: {YOLO_DATA_DIR}/images/test/

# Class names
names:
  0: ice
"""

with open('data-fined.yaml', 'w') as f:
    f.write(data_yaml_content)

print("YOLO data-fined.yaml file created.")

YOLO data-fined.yaml file created.


In [ ]:
# Part 4: Model Training

print("\nStarting YOLOv8 segmentation model training with augmentations...")

model = YOLO('yolov8n-seg.pt')

results = model.train(
    data="data-fined.yaml",
    epochs=30,
    imgsz=640,
    plots=True,
    fliplr=0.5,
    degrees=10,
    translate=0.1
)

print("Model training completed.")


Starting YOLOv8 segmentation model training with augmentations...
Ultralytics 8.3.196 🚀 Python-3.11.13 torch-2.8.0+cu128 CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data-fined.yaml, degrees=10, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=

/home/kit/.pyenv/versions/META/lib/python3.11/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Plotting labels to /home/kit/.pyenv/runs/segment/train6/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 66 weight(decay=0.0), 77 weight(decay=0.0005), 76 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /home/kit/.pyenv/runs/segment/train6
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size
       1/30         0G      1.587      3.588      2.846      1.756         28        640: 100% ━━━━━━━━━━━━ 30/30 0.2it/s 2:535.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 0.2it/s 28.9s7.9ss
                   all        154        242    0.00266      0.508     0.0782     0.0349    0.00264      

In [ ]:
print("\nEvaluating the trained model on the validation set...")

metrics = model.val()

print("\n--- Model Evaluation Metrics ---")
print(f"Mean Average Precision (mAP) at IoU 0.5: {metrics.seg.map50:.4f}")
print(f"Mean Average Precision (mAP) at IoU 0.5-0.95: {metrics.seg.map:.4f}")
print("-----------------------------------")


Evaluating the trained model on the validation set...
Ultralytics 8.3.196 🚀 Python-3.11.13 torch-2.8.0+cu128 CPU (AMD Ryzen 5 5600H with Radeon Graphics)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 695.3±462.7 MB/s, size: 44.8 KB)
val: Scanning /home/kit/Desktop/flood/yolo_icenet/labels/valid.cache... 152 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 154/154 220.1Kit/s 0.0s
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...
WARNING ⚠️ Limiting validation plots to first 50 items 

In [ ]:
print("\nRunning a prediction on a sample test image...")

results = model.predict(source=os.path.join(YOLO_DATA_DIR, 'images', 'test'), save=True, conf=0.25)



Running a prediction on a sample test image...

image 1/153 /home/kit/Desktop/flood/yolo_icenet/images/test/0622_1050_jpg.rf.866519451400deaaf48d165254404a9e.jpg: 640x640 3 ices, 102.8ms
image 2/153 /home/kit/Desktop/flood/yolo_icenet/images/test/0622_1063_jpg.rf.ca23de7eb7b4c04bf11fe1f6c10e7112.jpg: 640x640 1 ice, 72.7ms
image 3/153 /home/kit/Desktop/flood/yolo_icenet/images/test/0622_1218_jpg.rf.97f9bbe4335cfd5871c058b6f1c7d820.jpg: 640x640 5 ices, 73.6ms
image 4/153 /home/kit/Desktop/flood/yolo_icenet/images/test/0622_1307_jpg.rf.85d20ae7afc9dc875069b7df1e01df78.jpg: 640x640 8 ices, 82.1ms
image 5/153 /home/kit/Desktop/flood/yolo_icenet/images/test/0622_1633_jpg.rf.5812c1052f60b82f1fb2aa1f1c8195a0.jpg: 640x640 3 ices, 73.9ms
image 6/153 /home/kit/Desktop/flood/yolo_icenet/images/test/0622_16387_jpg.rf.204730bb4a05c34e83c410b1a676f56b.jpg: 640x640 2 ices, 78.4ms
image 7/153 /home/kit/Desktop/flood/yolo_icenet/images/test/0622_16431_jpg.rf.e31ec93cb2c12c592950d8ef145e2da5.jpg: 640x64

In [ ]:
path_to = model.export(format="torchscript", name="icenet_finetuned.pt")

print(f"Model exported successfully to {path_to}")

Ultralytics 8.3.196 🚀 Python-3.11.13 torch-2.8.0+cu128 CPU (AMD Ryzen 5 5600H with Radeon Graphics)

PyTorch: starting from '/home/kit/.pyenv/runs/segment/train6/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 37, 8400), (1, 32, 160, 160)) (6.5 MB)

TorchScript: starting export with torch 2.8.0+cu128...
TorchScript: export success ✅ 1.5s, saved as '/home/kit/.pyenv/runs/segment/train6/weights/best.torchscript' (12.9 MB)

Export complete (1.8s)
Results saved to /home/kit/.pyenv/runs/segment/train6/weights
Predict:         yolo predict task=segment model=/home/kit/.pyenv/runs/segment/train6/weights/best.torchscript imgsz=640  
Validate:        yolo val task=segment model=/home/kit/.pyenv/runs/segment/train6/weights/best.torchscript imgsz=640 data=data-fined.yaml  
Visualize:       https://netron.app
Model exported successfully to /home/kit/.pyenv/runs/segment/train6/weights/best.torchscript


## Predictions on Real Images of Astana

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import torch

model_path = '/content/drive/MyDrive/icenet/best.pt'
image_path = '/content/drive/MyDrive/icenet/astanaice/ast1.jpeg'
output_dir = '/content/drive/MyDrive/icenet/results'
output_image_path = os.path.join(output_dir, 'ice_detection_result.jpg')

In [ ]:
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используется устройство: {device}")

Используется устройство: cuda


In [ ]:
model = YOLO(model_path).to(device)

In [ ]:
image = cv2.imread(image_path)

In [ ]:
if image is None:
    print(f"Ошибка: Изображение не найдено по пути {image_path}. Убедитесь, что путь верный.")
else:
    resized_image = cv2.resize(image, (640, 640))

In [ ]:
results = model(resized_image, conf=0.50, verbose=True)


0: 640x640 1 ice, 9.0ms
Speed: 1.7ms preprocess, 9.0ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)


In [ ]:
result_image = results[0].plot()

cv2.imwrite(output_image_path, result_image)

print(f"Изображение с результатом успешно сохранено по пути: {output_image_path}")

Изображение с результатом успешно сохранено по пути: /content/drive/MyDrive/icenet/results/ice_detection_result.jpg
